In [ ]:

# ============================================================
# STEP 1: IMPORT LIBRARIES AND LOAD THE ORIGINAL DATASET
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

# Upload the original CSV file
uploaded = files.upload()

# Check that exactly one CSV file was uploaded
csv_files = [
    filename for filename in uploaded
    if filename.lower().endswith(".csv")
]

if len(csv_files) != 1:
    raise ValueError(
        f"Expected one CSV file, but found {len(csv_files)}. "
        "Please upload Dataset_ATS_v2.csv only."
    )

file_path = csv_files[0]

# Load the original dataset
df = pd.read_csv(file_path)

# Display the initial dataset information
print("DATASET LOADED SUCCESSFULLY")
print("=" * 50)
print("File name:", file_path)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nFIRST FIVE ROWS")
display(df.head())

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nDATA TYPES AND NON-NULL COUNTS")
df.info()


Saving Dataset_ATS_v2.csv to Dataset_ATS_v2.csv
DATASET LOADED SUCCESSFULLY
File name: Dataset_ATS_v2.csv
Number of rows: 7043
Number of columns: 10

FIRST FIVE ROWS


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No



COLUMN NAMES
['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']

DATA TYPES AND NON-NULL COUNTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   gender           7043 non-null   object
 1   SeniorCitizen    7043 non-null   int64 
 2   Dependents       7043 non-null   object
 3   tenure           7043 non-null   int64 
 4   PhoneService     7043 non-null   object
 5   MultipleLines    7043 non-null   object
 6   InternetService  7043 non-null   object
 7   Contract         7043 non-null   object
 8   MonthlyCharges   7043 non-null   int64 
 9   Churn            7043 non-null   object
dtypes: int64(3), object(7)
memory usage: 550.4+ KB


In [ ]:

# ============================================================
# STEP 2: AUDIT DATA QUALITY — NO DATA IS MODIFIED
# ============================================================

# 1. Confirm dataset structure and column names
print("DATASET STRUCTURE")
print("-" * 50)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())

# 2. Check standard missing values
print("\nSTANDARD MISSING VALUES")
print("-" * 50)
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})
display(missing_summary)

# 3. Check blank or whitespace-only text values
print("\nBLANK OR WHITESPACE-ONLY TEXT VALUES")
print("-" * 50)
text_columns = df.select_dtypes(include=["object", "string"]).columns

blank_counts = {}
for col in text_columns:
    blank_counts[col] = df[col].map(
        lambda value: isinstance(value, str) and value.strip() == ""
    ).sum()

display(pd.DataFrame.from_dict(
    blank_counts, orient="index", columns=["Blank_Count"]
))

# 4. Flag possible missing-value placeholders for review
# These are flagged, not automatically replaced.
possible_placeholders = {"na", "n/a", "null", "none", "unknown", "?"}

print("\nPOSSIBLE PLACEHOLDER VALUES — REVIEW REQUIRED")
print("-" * 50)

placeholder_rows = []
for col in text_columns:
    normalized = df[col].astype("string").str.strip().str.lower()
    counts = normalized[normalized.isin(possible_placeholders)].value_counts()

    for value, count in counts.items():
        placeholder_rows.append({
            "Column": col,
            "Value": value,
            "Count": int(count)
        })

if placeholder_rows:
    display(pd.DataFrame(placeholder_rows))
else:
    print("No listed placeholder values found.")

# 5. Inspect categories and their frequencies
print("\nCATEGORICAL VALUE COUNTS")
print("-" * 50)

for col in text_columns:
    print(f"\n{col}")
    display(df[col].value_counts(dropna=False).rename_axis("Value").to_frame("Count"))

# 6. Check numerical ranges and unusual values
print("\nNUMERICAL SUMMARY")
print("-" * 50)
numeric_columns = df.select_dtypes(include="number").columns
display(df[numeric_columns].describe().T)

print("\nNEGATIVE NUMERICAL VALUES")
print("-" * 50)
negative_counts = (df[numeric_columns] < 0).sum()
display(negative_counts.rename("Negative_Count").to_frame())

# 7. Check duplicate-looking rows
print("\nDUPLICATE-LOOKING RECORDS")
print("-" * 50)
duplicate_count = int(df.duplicated().sum())
print("Rows matching an earlier row across all columns:", duplicate_count)
print(
    "Percentage of rows matching an earlier row:",
    round(duplicate_count / len(df) * 100, 2), "%"
)

# 8. Inspect churn target distribution
print("\nCHURN TARGET DISTRIBUTION")
print("-" * 50)
if "Churn" in df.columns:
    churn_summary = pd.DataFrame({
        "Count": df["Churn"].value_counts(dropna=False),
        "Percentage": (
            df["Churn"].value_counts(normalize=True, dropna=False) * 100
        ).round(2)
    })
    display(churn_summary)
else:
    print("WARNING: Expected target column 'Churn' was not found.")

print("\nAUDIT COMPLETE — ORIGINAL DATAFRAME REMAINS UNCHANGED.")


DATASET STRUCTURE
--------------------------------------------------
Rows: 7043
Columns: 10
Duplicate column names: []

STANDARD MISSING VALUES
--------------------------------------------------


,Missing_Count,Missing_Percentage
gender,0,0.0
SeniorCitizen,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
Contract,0,0.0
MonthlyCharges,0,0.0
Churn,0,0.0



BLANK OR WHITESPACE-ONLY TEXT VALUES
--------------------------------------------------


,Blank_Count
gender,0
Dependents,0
PhoneService,0
MultipleLines,0
InternetService,0
Contract,0
Churn,0



POSSIBLE PLACEHOLDER VALUES — REVIEW REQUIRED
--------------------------------------------------
No listed placeholder values found.

CATEGORICAL VALUE COUNTS
--------------------------------------------------

gender


,Count
Value,
Male,3555
Female,3488


... (15 outputs left out)

,Negative_Count
SeniorCitizen,0
tenure,0
MonthlyCharges,0



DUPLICATE-LOOKING RECORDS
--------------------------------------------------
Rows matching an earlier row across all columns: 302
Percentage of rows matching an earlier row: 4.29 %

CHURN TARGET DISTRIBUTION
--------------------------------------------------


,Count,Percentage
Churn,,
No,5174,73.46
Yes,1869,26.54



AUDIT COMPLETE — ORIGINAL DATAFRAME REMAINS UNCHANGED.


In [ ]:

# ============================================================
# STEP 3: VALIDATE VALUES AND CREATE A WORKING COPY
# ============================================================

# Expected values based on the dataset structure and project context
allowed_categories = {
    "gender": {"Female", "Male"},
    "Dependents": {"No", "Yes"},
    "PhoneService": {"No", "Yes"},
    "MultipleLines": {"No", "Yes"},
    "InternetService": {"DSL", "Fiber optic"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "Churn": {"No", "Yes"}
}

checks = []

# Check categorical values
for column, allowed_values in allowed_categories.items():
    actual_values = set(df[column].dropna().unique())
    unexpected = sorted(actual_values - allowed_values)

    checks.append({
        "Check": f"{column}: allowed categories",
        "Result": "PASS" if not unexpected else "REVIEW",
        "Details": (
            "All values are expected"
            if not unexpected
            else f"Unexpected values: {unexpected}"
        )
    })

# Check binary senior-citizen indicator
senior_values = set(df["SeniorCitizen"].dropna().unique())
checks.append({
    "Check": "SeniorCitizen: binary values",
    "Result": "PASS" if senior_values.issubset({0, 1}) else "REVIEW",
    "Details": f"Observed values: {sorted(senior_values)}"
})

# Check tenure range and integer values
tenure_valid = (
    df["tenure"].notna().all()
    and df["tenure"].between(0, 72).all()
    and (df["tenure"] % 1 == 0).all()
)
checks.append({
    "Check": "tenure: whole months from 0 to 72",
    "Result": "PASS" if tenure_valid else "REVIEW",
    "Details": f"Observed range: {df['tenure'].min()} to {df['tenure'].max()}"
})

# Check monthly charges
charges_valid = (
    df["MonthlyCharges"].notna().all()
    and (df["MonthlyCharges"] > 0).all()
)
checks.append({
    "Check": "MonthlyCharges: positive values",
    "Result": "PASS" if charges_valid else "REVIEW",
    "Details": (
        f"Observed range: {df['MonthlyCharges'].min()} "
        f"to {df['MonthlyCharges'].max()}"
    )
})

validation_report = pd.DataFrame(checks)

print("DATA VALIDATION REPORT")
display(validation_report)

# Create a separate working copy; preserve the original df
df_clean = df.copy()

# Remove leading/trailing whitespace from text fields, if present.
# This does not remove rows or change category meanings.
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[column] = df_clean[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

print("\nCLEANING SUMMARY")
print("Original rows:", len(df))
print("Working-copy rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print("Original dataset preserved:", df.equals(df_clean))


DATA VALIDATION REPORT


,Check,Result,Details
0,gender: allowed categories,PASS,All values are expected
1,Dependents: allowed categories,PASS,All values are expected
2,PhoneService: allowed categories,PASS,All values are expected
3,MultipleLines: allowed categories,PASS,All values are expected
4,InternetService: allowed categories,PASS,All values are expected
5,Contract: allowed categories,PASS,All values are expected
6,Churn: allowed categories,PASS,All values are expected
7,SeniorCitizen: binary values,PASS,"Observed values: [np.int64(0), np.int64(1)]"
8,tenure: whole months from 0 to 72,PASS,Observed range: 0 to 72
9,MonthlyCharges: positive values,PASS,Observed range: 18 to 119



CLEANING SUMMARY
Original rows: 7043
Working-copy rows: 7043
Rows removed: 0
Original dataset preserved: True


In [ ]:

# ============================================================
# STEP 4: REVIEW FEATURE VALUES AND DEFINE ANALYSIS INPUTS
# ============================================================

# Review service combinations before feature engineering
print("PHONE SERVICE AND MULTIPLE LINES")
display(pd.crosstab(
    df_clean["PhoneService"],
    df_clean["MultipleLines"],
    margins=True
))

# Define the target and candidate input features
target_column = "Churn"

candidate_features = [
    "gender",
    "SeniorCitizen",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "Contract",
    "MonthlyCharges"
]

# Document the feature roles
feature_plan = pd.DataFrame([
    ["gender", "Demographic", "Candidate input"],
    ["SeniorCitizen", "Demographic", "Candidate input; binary indicator"],
    ["Dependents", "Demographic", "Candidate input"],
    ["tenure", "Customer relationship", "Candidate input; months with provider"],
    ["PhoneService", "Service usage", "Candidate input"],
    ["MultipleLines", "Service usage", "Candidate input"],
    ["InternetService", "Service usage", "Candidate input"],
    ["Contract", "Contract type", "Candidate input"],
    ["MonthlyCharges", "Billing", "Candidate input; monthly charge amount"],
    ["Churn", "Prediction target", "Target only; exclude from clustering inputs"]
], columns=["Column", "Feature Group", "Planned Use"])

print("\nFEATURE PLAN")
display(feature_plan)

# Verify all planned columns exist
missing_columns = [
    column for column in candidate_features + [target_column]
    if column not in df_clean.columns
]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("\nFeature-plan check: PASS")
print("Candidate input features:", len(candidate_features))
print("Prediction target:", target_column)


PHONE SERVICE AND MULTIPLE LINES


MultipleLines,No,Yes,All
PhoneService,,,
No,422,260,682
Yes,3650,2711,6361
All,4072,2971,7043



FEATURE PLAN


,Column,Feature Group,Planned Use
0,gender,Demographic,Candidate input
1,SeniorCitizen,Demographic,Candidate input; binary indicator
2,Dependents,Demographic,Candidate input
3,tenure,Customer relationship,Candidate input; months with provider
4,PhoneService,Service usage,Candidate input
5,MultipleLines,Service usage,Candidate input
6,InternetService,Service usage,Candidate input
7,Contract,Contract type,Candidate input
8,MonthlyCharges,Billing,Candidate input; monthly charge amount
9,Churn,Prediction target,Target only; exclude from clustering inputs



Feature-plan check: PASS
Candidate input features: 9
Prediction target: Churn


In [ ]:

# ============================================================
# STEP 5: ENCODE CATEGORICAL INPUT FEATURES
# ============================================================

# Separate the target from input features
X_raw = df_clean[candidate_features].copy()
y = df_clean["Churn"].map({"No": 0, "Yes": 1}).rename("Churn")

if y.isna().any():
    raise ValueError("Unexpected values found in the Churn target.")

# Convert categorical input columns into one-hot encoded columns.
# drop_first=True avoids redundant reference columns for binary categories.
categorical_features = X_raw.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

X_encoded = pd.get_dummies(
    X_raw,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("ENCODING SUMMARY")
print("Original input feature count:", len(candidate_features))
print("Categorical columns encoded:", categorical_features)
print("Encoded input shape:", X_encoded.shape)
print("Target shape:", y.shape)

print("\nENCODED DATA PREVIEW")
display(X_encoded.head())

print("\nTARGET DISTRIBUTION")
display(
    y.value_counts()
    .rename(index={0: "No churn", 1: "Churn"})
    .to_frame("Count")
)

# Confirm the target is not part of the encoded input
assert "Churn" not in X_encoded.columns
assert len(X_encoded) == len(y)
assert not X_encoded.isna().any().any()

print("\nValidation passed: no target leakage into encoded inputs,")
print("no missing encoded values, and row alignment is preserved.")


ENCODING SUMMARY
Original input feature count: 9
Categorical columns encoded: ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract']
Encoded input shape: (7043, 10)
Target shape: (7043,)

ENCODED DATA PREVIEW


,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,Contract_One year,Contract_Two year
0,0,1,25,0,0,0,0,0,0,0
1,0,41,25,1,0,1,0,0,1,0
2,0,52,19,0,1,1,0,0,0,0
3,0,1,76,0,0,1,0,0,1,0
4,0,67,51,1,0,1,0,1,0,0



TARGET DISTRIBUTION


,Count
Churn,
No churn,5174
Churn,1869



Validation passed: no target leakage into encoded inputs,
no missing encoded values, and row alignment is preserved.


In [ ]:

# ============================================================
# STEP 6: SCALE CLUSTERING FEATURES AND PREPARE HANDOVER FILES
# ============================================================

from sklearn.preprocessing import StandardScaler
from pathlib import Path

# 1. Create clustering inputs without the target
X_clustering = X_encoded.copy()

# 2. Scale the clustering inputs
clustering_scaler = StandardScaler()
X_clustering_scaled = pd.DataFrame(
    clustering_scaler.fit_transform(X_clustering),
    columns=X_clustering.columns,
    index=X_clustering.index
)

# 3. Validate the scaled data
scaling_summary = pd.DataFrame({
    "Mean_After_Scaling": X_clustering_scaled.mean().round(4),
    "Std_After_Scaling": X_clustering_scaled.std(ddof=0).round(4),
    "Minimum": X_clustering_scaled.min().round(4),
    "Maximum": X_clustering_scaled.max().round(4)
})

print("CLUSTERING SCALING CHECK")
display(scaling_summary)

# 4. Prepare a clear handover folder
output_dir = Path("ACS_Churn_Stage2_Outputs")
output_dir.mkdir(exist_ok=True)

# 5. Export predictive inputs without scaling
X_encoded.to_csv(
    output_dir / "predictive_inputs_encoded_unscaled.csv",
    index=False
)

# 6. Export clustering inputs after scaling
X_clustering_scaled.to_csv(
    output_dir / "clustering_inputs_scaled.csv",
    index=False
)

# 7. Export the target separately
y.to_csv(
    output_dir / "churn_target.csv",
    index=False
)

# 8. Export clustering inputs before scaling for traceability
X_clustering.to_csv(
    output_dir / "clustering_inputs_encoded_unscaled.csv",
    index=False
)

print("\nEXPORTED FILES")
for file_path in sorted(output_dir.iterdir()):
    print(f"{file_path.name}: {file_path.stat().st_size:,} bytes")

print("\nSHAPE VALIDATION")
print("Predictive inputs:", X_encoded.shape)
print("Unscaled clustering inputs:", X_clustering.shape)
print("Scaled clustering inputs:", X_clustering_scaled.shape)
print("Churn target:", y.shape)

assert X_encoded.shape[0] == len(y)
assert X_clustering_scaled.shape == X_clustering.shape
assert "Churn" not in X_clustering_scaled.columns
assert np.isfinite(X_clustering_scaled.to_numpy()).all()

print("\nAll output checks passed.")

# ============================================================
# STEP 6: SCALE CLUSTERING FEATURES AND PREPARE HANDOVER FILES
# ============================================================

from sklearn.preprocessing import StandardScaler
from pathlib import Path

# 1. Create clustering inputs without the target
X_clustering = X_encoded.copy()

# 2. Scale the clustering inputs
clustering_scaler = StandardScaler()
X_clustering_scaled = pd.DataFrame(
    clustering_scaler.fit_transform(X_clustering),
    columns=X_clustering.columns,
    index=X_clustering.index
)

# 3. Validate the scaled data
scaling_summary = pd.DataFrame({
    "Mean_After_Scaling": X_clustering_scaled.mean().round(4),
    "Std_After_Scaling": X_clustering_scaled.std(ddof=0).round(4),
    "Minimum": X_clustering_scaled.min().round(4),
    "Maximum": X_clustering_scaled.max().round(4)
})

print("CLUSTERING SCALING CHECK")
display(scaling_summary)

# 4. Prepare a clear handover folder
output_dir = Path("ACS_Churn_Stage2_Outputs")
output_dir.mkdir(exist_ok=True)

# 5. Export predictive inputs without scaling
X_encoded.to_csv(
    output_dir / "predictive_inputs_encoded_unscaled.csv",
    index=False
)

# 6. Export clustering inputs after scaling
X_clustering_scaled.to_csv(
    output_dir / "clustering_inputs_scaled.csv",
    index=False
)

# 7. Export the target separately
y.to_csv(
    output_dir / "churn_target.csv",
    index=False
)

# 8. Export clustering inputs before scaling for traceability
X_clustering.to_csv(
    output_dir / "clustering_inputs_encoded_unscaled.csv",
    index=False
)

print("\nEXPORTED FILES")
for file_path in sorted(output_dir.iterdir()):
    print(f"{file_path.name}: {file_path.stat().st_size:,} bytes")

print("\nSHAPE VALIDATION")
print("Predictive inputs:", X_encoded.shape)
print("Unscaled clustering inputs:", X_clustering.shape)
print("Scaled clustering inputs:", X_clustering_scaled.shape)
print("Churn target:", y.shape)

assert X_encoded.shape[0] == len(y)
assert X_clustering_scaled.shape == X_clustering.shape
assert "Churn" not in X_clustering_scaled.columns
assert np.isfinite(X_clustering_scaled.to_numpy()).all()

print("\nAll output checks passed.")


CLUSTERING SCALING CHECK


,Mean_After_Scaling,Std_After_Scaling,Minimum,Maximum
SeniorCitizen,-0.0,1.0,-0.4399,2.2732
tenure,-0.0,1.0,-1.3182,1.6137
MonthlyCharges,-0.0,1.0,-1.5540,1.8027
gender_Male,-0.0,1.0,-1.0096,0.9905
Dependents_Yes,0.0,1.0,-0.6540,1.5290
PhoneService_Yes,0.0,1.0,-3.0540,0.3274
MultipleLines_Yes,-0.0,1.0,-0.8542,1.1707
InternetService_Fiber optic,-0.0,1.0,-0.8857,1.1291
Contract_One year,-0.0,1.0,-0.5142,1.9446
Contract_Two year,-0.0,1.0,-0.5630,1.7763



EXPORTED FILES
churn_target.csv: 14,092 bytes
clustering_inputs_encoded_unscaled.csv: 154,219 bytes
clustering_inputs_scaled.csv: 1,374,679 bytes
predictive_inputs_encoded_unscaled.csv: 154,219 bytes

SHAPE VALIDATION
Predictive inputs: (7043, 10)
Unscaled clustering inputs: (7043, 10)
Scaled clustering inputs: (7043, 10)
Churn target: (7043,)

All output checks passed.
CLUSTERING SCALING CHECK


,Mean_After_Scaling,Std_After_Scaling,Minimum,Maximum
SeniorCitizen,-0.0,1.0,-0.4399,2.2732
tenure,-0.0,1.0,-1.3182,1.6137
MonthlyCharges,-0.0,1.0,-1.5540,1.8027
gender_Male,-0.0,1.0,-1.0096,0.9905
Dependents_Yes,0.0,1.0,-0.6540,1.5290
PhoneService_Yes,0.0,1.0,-3.0540,0.3274
MultipleLines_Yes,-0.0,1.0,-0.8542,1.1707
InternetService_Fiber optic,-0.0,1.0,-0.8857,1.1291
Contract_One year,-0.0,1.0,-0.5142,1.9446
Contract_Two year,-0.0,1.0,-0.5630,1.7763



EXPORTED FILES
churn_target.csv: 14,092 bytes
clustering_inputs_encoded_unscaled.csv: 154,219 bytes
clustering_inputs_scaled.csv: 1,374,679 bytes
predictive_inputs_encoded_unscaled.csv: 154,219 bytes

SHAPE VALIDATION
Predictive inputs: (7043, 10)
Unscaled clustering inputs: (7043, 10)
Scaled clustering inputs: (7043, 10)
Churn target: (7043,)

All output checks passed.


In [ ]:

# ============================================================
# STEP 7: FINAL QUALITY ASSURANCE AND HANDOVER PACKAGE
# ============================================================

import joblib
import zipfile
from pathlib import Path

# Save the fitted clustering scaler for reproducibility
joblib.dump(
    clustering_scaler,
    output_dir / "clustering_standard_scaler.joblib"
)

# Final quality checks
quality_checks = {
    "Original row count preserved": len(df_clean) == 7043,
    "Encoded inputs contain 7043 rows": len(X_encoded) == len(df_clean),
    "Target contains 7043 values": len(y) == len(df_clean),
    "Target excluded from clustering inputs": "Churn" not in X_clustering_scaled.columns,
    "Encoded inputs have no missing values": not X_encoded.isna().any().any(),
    "Scaled inputs contain only finite values": np.isfinite(X_clustering_scaled.to_numpy()).all(),
    "Churn target contains only 0 and 1": set(y.unique()).issubset({0, 1}),
}

quality_report = pd.DataFrame(
    [{"Check": name, "Result": "PASS" if passed else "FAIL"}
     for name, passed in quality_checks.items()]
)

print("FINAL QUALITY ASSURANCE")
display(quality_report)

if not all(quality_checks.values()):
    raise ValueError("At least one final quality check failed.")

# Create a concise handover README
readme = """
ACS CUSTOMER CHURN ANALYSIS
Stage 1–2 Data Engineering Handover
Role: Data Engineer — Preprocessing and Feature Engineering

SOURCE DATA
Dataset_ATS_v2.csv
Rows: 7,043
Columns: 10

WORK COMPLETED
1. Loaded and inspected the original dataset.
2. Checked standard missing values, blank text, listed placeholders,
   categorical values, numeric ranges, negative values and duplicates.
3. Validated the observed categories and basic numerical constraints.
4. Created a working copy; no rows were removed.
5. Encoded categorical input features using one-hot encoding.
6. Converted Churn to a separate binary target: No = 0, Yes = 1.
7. Created scaled clustering inputs and unscaled predictive inputs.
8. Ran final shape, target-separation and numerical-validity checks.

FILES
predictive_inputs_encoded_unscaled.csv
    Encoded input features for predictive modelling. Split the data
    before fitting any scaler or other learned preprocessing.

clustering_inputs_encoded_unscaled.csv
    Encoded input features before scaling.

clustering_inputs_scaled.csv
    StandardScaler-transformed inputs for exploratory clustering.
    The scaler was fitted on the full dataset; use care if evaluating
    clustering on held-out data.

churn_target.csv
    Binary target aligned row-by-row with the exported input files.
    No = 0; Yes = 1.

clustering_standard_scaler.joblib
    Fitted scaler used to create clustering_inputs_scaled.csv.

IMPORTANT LIMITATIONS AND REVIEW ITEMS
- The dataset has no customer ID. There were 302 rows matching an
  earlier row across all columns (4.29%). They were retained because
  identical attributes do not prove that records represent the same
  customer.
- The PhoneService/MultipleLines crosstab showed combinations that
  may need clarification, including PhoneService = No with
  MultipleLines = Yes. No values were changed based on assumptions.
- No missing values, blank text values, listed placeholders or
  negative numeric values were detected by the checks performed.
  These checks do not prove that every source value is factually correct.
- Churn was excluded from clustering inputs to avoid using the outcome
  to construct the segments.
- Predictive inputs are unscaled so the analyst can split the data
  first and fit learned preprocessing on training data only.
- The current feature set is a candidate set, not proof that every
  feature is useful. The modelling team should confirm suitability
  against the project requirements and source data dictionary.

TARGET DISTRIBUTION
No churn: 5,174 (73.46%)
Churn: 1,869 (26.54%)

HANDOVER
Keep each input file and churn_target.csv in the same row order.
Do not sort or filter one file independently of the others.
Confirm the service-field definitions and duplicate-record policy
with the project team before making further changes.
"""

readme_path = output_dir / "README_Data_Engineering_Handover.txt"
readme_path.write_text(readme.strip(), encoding="utf-8")

# Package all outputs into one ZIP file
zip_path = Path("ACS_Customer_Churn_Stage1_Stage2_Handover.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in sorted(output_dir.iterdir()):
        if file_path.is_file():
            zipf.write(file_path, arcname=file_path.name)

print("\nHANDOVER PACKAGE CONTENTS")
for file_path in sorted(output_dir.iterdir()):
    if file_path.is_file():
        print(f"- {file_path.name} ({file_path.stat().st_size:,} bytes)")

print("\nZIP PACKAGE CREATED:", zip_path.name)
print("ZIP SIZE:", f"{zip_path.stat().st_size:,} bytes")
print("All final quality checks passed.")


FINAL QUALITY ASSURANCE


,Check,Result
0,Original row count preserved,PASS
1,Encoded inputs contain 7043 rows,PASS
2,Target contains 7043 values,PASS
3,Target excluded from clustering inputs,PASS
4,Encoded inputs have no missing values,PASS
5,Scaled inputs contain only finite values,PASS
6,Churn target contains only 0 and 1,PASS



HANDOVER PACKAGE CONTENTS
- README_Data_Engineering_Handover.txt (2,829 bytes)
- churn_target.csv (14,092 bytes)
- clustering_inputs_encoded_unscaled.csv (154,219 bytes)
- clustering_inputs_scaled.csv (1,374,679 bytes)
- clustering_standard_scaler.joblib (1,287 bytes)
- predictive_inputs_encoded_unscaled.csv (154,219 bytes)

ZIP PACKAGE CREATED: ACS_Customer_Churn_Stage1_Stage2_Handover.zip
ZIP SIZE: 142,637 bytes
All final quality checks passed.


In [ ]:

from google.colab import files
files.download("ACS_Customer_Churn_Stage1_Stage2_Handover.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>